In [ ]:
import requests
import pandas as pd
import os
import datetime
import numpy  as np
from sqlalchemy import create_engine, text

## Étape 1 — Extraction / Bronze
Gérer les erreurs lors des appels API : timeout, erreurs HTTP, réponses invalides, etc.

1. Récupérer le dataset des villes marocaines.

In [3]:
ma_url = "https://simplemaps.com/data/ma-cities"
ma_json_url = "https://simplemaps.com/static/data/country-cities/ma/ma.json"

try:
    response = requests.get(url=ma_json_url, timeout=60)
    response.raise_for_status()
    ma_df = pd.DataFrame(data=response.json() or [])
    print(ma_df)
except requests.exceptions.Timeout:
    print("L'API a pris plus d'une minute pour répondre.")
except requests.exceptions.HTTPError as e:
    print(f"Erreur HTTP: {e}")
except requests.exceptions.JSONDecodeError:
    print("Format json invalide.")
# La base des exceptions sourvenues lors d'un request.
except requests.exceptions.RequestException as r:
    print(f"Request Exception: {r}")

                  city      lat       lng  country iso2  \
0           Casablanca  33.5992   -7.6200  Morocco   MA   
1              Tangier  35.7767   -5.8039  Morocco   MA   
2                  Fès  34.0433   -5.0033  Morocco   MA   
3            Marrakech  31.6295   -7.9811  Morocco   MA   
4                 Sale  34.0500   -6.8167  Morocco   MA   
..                 ...      ...       ...      ...  ...   
115        Oulad Yaïch  32.4167   -6.3333  Morocco   MA   
116  Zawyat ech Cheïkh  32.6541   -5.9214  Morocco   MA   
117       Imi-n-Tanout  31.1770   -8.8504  Morocco   MA   
118        Sebt Gzoula  32.1219   -9.0889  Morocco   MA   
119           Tifariti  26.1580  -10.5670  Morocco   MA   

                    admin_name  capital population population_proper  
0            Casablanca-Settat    admin    3950000           3215935  
1    Tanger-Tétouan-Al Hoceïma    admin    1275428           1275428  
2                   Fès-Meknès    admin    1167842           1167842  
3      

2. Utiliser les coordonnées des villes pour interroger l'API Open-Meteo.

In [4]:
meteo_data = []
meteo_url = "https://api.open-meteo.com/v1/forecast"

for lat, lng in ma_df[['lat', 'lng']].to_numpy():
    meteo_params = {
        "latitude": lat,
        "longitude": lng,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "precipitation_probability_max",
            "wind_speed_10m_max",
            "wind_gusts_10m_max"
        ],
        "current": "weather_code",
        "forecast_days": 3
    }
    try:
        response = requests.get(url=meteo_url, params=meteo_params, timeout=300)
        response.raise_for_status()
        meteo_data.append(response.json())
    except requests.exceptions.Timeout:
        print("L'API a pris plus de cinqs minutes pour répondre.")
        break
    except requests.exceptions.HTTPError as e:
        print(f"Erreur HTTP: {e}")
        break
    except requests.exceptions.JSONDecodeError:
        print("Format json invalide.")
        break
    except requests.exceptions.RequestException as r:
        print(f"Request Exception: {r}")
        break

3. Récupérer les prévisions météorologiques quotidiennes des prochains jours.

In [5]:
meteo_df = pd.DataFrame(data=meteo_data)
meteo_df

,latitude,longitude,generationtime_ms,utc_offset_seconds,timezone,timezone_abbreviation,elevation,current_units,current,daily_units,daily
0,33.56250,-7.625000,0.109673,0,GMT,GMT,23.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
1,35.75000,-5.812500,0.108957,0,GMT,GMT,30.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
2,34.06250,-5.000000,0.119209,0,GMT,GMT,389.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
3,31.62500,-8.000000,0.102758,0,GMT,GMT,469.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
4,34.00000,-6.812500,0.102520,0,GMT,GMT,27.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
...,...,...,...,...,...,...,...,...,...,...,...
115,32.43750,-6.312500,0.442147,0,GMT,GMT,503.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
116,32.68750,-5.937500,0.460029,0,GMT,GMT,657.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
117,31.18750,-8.875000,0.739813,0,GMT,GMT,863.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
118,32.12500,-9.062500,0.140309,0,GMT,GMT,173.0,"{'time': 'iso8601', 'interval': 'seconds', 'we...","{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': 'iso8601', 'temperature_2m_max': '°C'...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."


4. Conserver les données brutes dans bronze/.

In [6]:
try:
    os.mkdir(path="./bronze")
    ma_df.to_csv(path_or_buf="./bronze/ma.csv", index=False)
    meteo_df.to_csv(path_or_buf="./bronze/meteo.csv", index=False)
except FileExistsError:
    ma_df.to_csv(path_or_buf="./bronze/ma.csv", index=False)
    meteo_df.to_csv(path_or_buf="./bronze/meteo.csv", index=False)
except Exception as e:
    print(e)

In [ ]:
# meteo_params_2 = {
#     "latitude": ma_df["lat"].tolist(),
#     "longitude": ma_df["lng"].tolist(),
#     "daily": [
#         "temperature_2m_max",
#         "temperature_2m_min",
#         "precipitation_sum",
#         "precipitation_probability_max",
#         "wind_speed_10m_max",
#         "wind_gusts_10m_max"
#     ],
#     "current": "weather_code",
#     "forecast_days": 3
# }
# response2 = requests.get(
#     url=meteo_url,
#     params=meteo_params_2,
#     timeout=300
# )
# response2.raise_for_status()
# response2.json()

## Étape 2 — Nettoyage / Silver

1. standardiser les types et les dates.

In [7]:
ma_df.dtypes

city                 str
lat                  str
lng                  str
country              str
iso2                 str
admin_name           str
capital              str
population           str
population_proper    str
dtype: object

In [8]:
ma_df["lat"] = ma_df["lat"].astype(float)
ma_df["lng"] = ma_df["lng"].astype(float)
ma_df["population"] = ma_df["population"].astype(float)
ma_df["population_proper"] = ma_df["population_proper"].astype(float)
ma_df.dtypes

city                     str
lat                  float64
lng                  float64
country                  str
iso2                     str
admin_name               str
capital                  str
population           float64
population_proper    float64
dtype: object

In [8]:
ma_df.columns

Index(['city', 'lat', 'lng', 'country', 'iso2', 'admin_name', 'capital',
       'population', 'population_proper'],
      dtype='str')

In [9]:
ma_df.drop(labels=['iso2','admin_name', 'capital', 'population', 'population_proper'], axis=1, inplace=True)
ma_df.head()

,city,lat,lng,country
0,Casablanca,33.5992,-7.6200,Morocco
1,Tangier,35.7767,-5.8039,Morocco
2,Fès,34.0433,-5.0033,Morocco
3,Marrakech,31.6295,-7.9811,Morocco
4,Sale,34.0500,-6.8167,Morocco


In [10]:
meteo_df.dtypes

latitude                 float64
longitude                float64
generationtime_ms        float64
utc_offset_seconds         int64
timezone                     str
timezone_abbreviation        str
elevation                float64
current_units             object
current                   object
daily_units               object
daily                     object
dtype: object

In [11]:
meteo_df.drop(
    labels=[
        "generationtime_ms", 
        "utc_offset_seconds", 
        "timezone", 
        "timezone_abbreviation",
        "current_units", 
        "daily_units"
    ], 
    axis=1, 
    inplace=True
)
meteo_df

,latitude,longitude,elevation,current,daily
0,33.56250,-7.625000,23.0,"{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
1,35.75000,-5.812500,30.0,"{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
2,34.06250,-5.000000,389.0,"{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
3,31.62500,-8.000000,469.0,"{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
4,34.00000,-6.812500,27.0,"{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
...,...,...,...,...,...
115,32.43750,-6.312500,503.0,"{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
116,32.68750,-5.937500,657.0,"{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
117,31.18750,-8.875000,863.0,"{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."
118,32.12500,-9.062500,173.0,"{'time': '2026-09-20T19:15', 'interval': 900, ...","{'time': ['2026-09-20', '2026-09-21', '2026-09..."


In [12]:
current_df = pd.DataFrame(
    data=meteo_df["current"].to_list(), 
    index=meteo_df["current"].index
)
current_df.head(1)

,time,interval,weather_code
0,2026-09-20T19:15,900,0


In [13]:
current_df.dtypes

time              str
interval        int64
weather_code    int64
dtype: object

In [13]:
current_df["time"] = pd.to_datetime(current_df.time)
current_df.dtypes

time            datetime64[us]
interval                 int64
weather_code             int64
dtype: object

In [14]:
daily_df = pd.DataFrame(
    data=meteo_df["daily"].to_list(), 
    index=meteo_df["daily"].index
)
daily_df.head(1)

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max
0,"[2026-09-20, 2026-09-21, 2026-09-22]","[30.3, 31.3, 29.8]","[20.5, 21.6, 19.8]","[0.0, 0.0, 0.0]","[0, 0, 0]","[10.1, 11.5, 9.6]","[29.5, 34.2, 28.8]"


In [15]:
daily_df.columns.to_list()

['time',
 'temperature_2m_max',
 'temperature_2m_min',
 'precipitation_sum',
 'precipitation_probability_max',
 'wind_speed_10m_max',
 'wind_gusts_10m_max']

In [16]:
daily_df_exploded = daily_df.explode(daily_df.columns.to_list())

In [17]:
daily_df_exploded.head(4)

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max
0,2026-09-20,30.3,20.5,0.0,0,10.1,29.5
0,2026-09-21,31.3,21.6,0.0,0,11.5,34.2
0,2026-09-22,29.8,19.8,0.0,0,9.6,28.8
1,2026-09-20,27.7,23.7,0.0,0,33.8,78.8


In [18]:
daily_df_exploded.dtypes

time                                str
temperature_2m_max               object
temperature_2m_min               object
precipitation_sum                object
precipitation_probability_max    object
wind_speed_10m_max               object
wind_gusts_10m_max               object
dtype: object

In [19]:
daily_df_exploded.precipitation_probability_max.unique()

array([0, 18, 3, 20, 2, 8, 21, 5, 45, 33, 15, 25, 10, 4, 13, 1, 23, 30,
       38], dtype=object)

In [20]:
# date in iso format 'yyyy-mm-dd'
daily_df_exploded.time = pd.to_datetime(daily_df_exploded.time)
daily_df_exploded.temperature_2m_max = daily_df_exploded.temperature_2m_max.astype(float)
daily_df_exploded.temperature_2m_min = daily_df_exploded.temperature_2m_min.astype(float)
daily_df_exploded.precipitation_sum = daily_df_exploded.precipitation_sum.astype(float)
daily_df_exploded.precipitation_probability_max = (
    daily_df_exploded
    .precipitation_probability_max
    .astype(int)
)
daily_df_exploded.wind_speed_10m_max = daily_df_exploded.wind_speed_10m_max.astype(float)
daily_df_exploded.wind_gusts_10m_max = daily_df_exploded.wind_gusts_10m_max.astype(float)

daily_df_exploded.dtypes

time                             datetime64[us]
temperature_2m_max                      float64
temperature_2m_min                      float64
precipitation_sum                       float64
precipitation_probability_max             int64
wind_speed_10m_max                      float64
wind_gusts_10m_max                      float64
dtype: object

In [21]:
meteo_df.drop(["current", "daily"], axis=1, inplace=True)
meteo_df.columns

Index(['latitude', 'longitude', 'elevation'], dtype='str')

In [22]:
daily_current_df = (
    daily_df_exploded.merge(
        current_df["weather_code"], 
        left_index=True, 
        right_index=True,
        suffixes=("_daily", "_current")
    )
)
daily_current_df.head(10)

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code
0,2026-09-20,30.3,20.5,0.0,0,10.1,29.5,0
0,2026-09-21,31.3,21.6,0.0,0,11.5,34.2,0
0,2026-09-22,29.8,19.8,0.0,0,9.6,28.8,0
1,2026-09-20,27.7,23.7,0.0,0,33.8,78.8,0
1,2026-09-21,28.6,22.8,0.0,0,27.7,64.1,0
1,2026-09-22,29.4,21.3,0.0,0,23.5,52.9,0
2,2026-09-20,35.7,21.9,0.0,0,14.5,37.4,0
2,2026-09-21,35.5,22.3,0.0,0,11.4,29.9,0
2,2026-09-22,34.3,20.5,0.0,0,10.0,25.9,0
3,2026-09-20,37.4,23.6,0.0,18,11.6,34.9,3


In [24]:
daily_current_df.shape

(360, 8)

In [25]:
meteo_df.shape

(120, 3)

In [23]:
meteo_df = (
    meteo_df.merge(
        daily_current_df,
        left_index=True, 
        right_index=True,
        suffixes=("_meteo", "_daily")
    )
)
meteo_df.shape

(360, 11)

In [27]:
meteo_df.head()

,latitude,longitude,elevation,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code
0,33.5625,-7.6250,23.0,2026-09-20,29.6,20.5,0.0,0,10.2,29.2,2
0,33.5625,-7.6250,23.0,2026-09-21,30.7,20.8,0.0,0,10.9,32.0,2
0,33.5625,-7.6250,23.0,2026-09-22,29.3,20.3,0.0,0,10.3,29.5,2
1,35.7500,-5.8125,30.0,2026-09-20,28.1,23.7,0.0,0,31.7,75.6,1
1,35.7500,-5.8125,30.0,2026-09-21,28.6,23.0,0.0,0,27.1,63.4,1


In [28]:
meteo_df.dtypes

latitude                                float64
longitude                               float64
elevation                               float64
time                             datetime64[us]
temperature_2m_max                      float64
temperature_2m_min                      float64
precipitation_sum                       float64
precipitation_probability_max             int64
wind_speed_10m_max                      float64
wind_gusts_10m_max                      float64
weather_code                              int64
dtype: object

2. contrôler la qualité des données.

In [24]:
ma_df[ma_df.duplicated() == True]

,city,lat,lng,country


In [30]:
ma_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   city     120 non-null    str    
 1   lat      120 non-null    float64
 2   lng      120 non-null    float64
 3   country  120 non-null    str    
dtypes: float64(2), str(2)
memory usage: 3.9 KB


In [25]:
ma_df.describe()

,lat,lng
count,120.000000,120.000000
mean,32.722032,-6.837944
std,2.148482,2.408281
min,23.716700,-15.950000
25%,31.568600,-8.357275
50%,33.227200,-6.694650
75%,34.184950,-5.525775
max,35.841400,-1.911400


In [26]:
meteo_df[meteo_df.duplicated() == True]

,latitude,longitude,elevation,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code


In [33]:
meteo_df.info()

<class 'pandas.DataFrame'>
Index: 360 entries, 0 to 119
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   latitude                       360 non-null    float64       
 1   longitude                      360 non-null    float64       
 2   elevation                      360 non-null    float64       
 3   time                           360 non-null    datetime64[us]
 4   temperature_2m_max             360 non-null    float64       
 5   temperature_2m_min             360 non-null    float64       
 6   precipitation_sum              360 non-null    float64       
 7   precipitation_probability_max  360 non-null    int64         
 8   wind_speed_10m_max             360 non-null    float64       
 9   wind_gusts_10m_max             360 non-null    float64       
 10  weather_code                   360 non-null    int64         
dtypes: datetime64[us](1), float64(8), i

In [27]:
meteo_df.drop("time", axis=1).describe()

,latitude,longitude,elevation,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code
count,360.000000,360.000000,360.000000,360.000000,360.000000,360.000000,360.000000,360.000000,360.000000,360.000000
mean,32.720886,-6.836718,366.025000,33.017222,20.681111,0.064444,2.277778,17.749167,35.133333,2.958333
std,2.139102,2.402353,375.567592,4.295283,2.306906,0.381117,6.504813,5.225889,9.125730,9.889131
min,23.725834,-15.966187,2.000000,21.700000,13.900000,0.000000,0.000000,7.600000,15.800000,0.000000
25%,31.562500,-8.343750,52.750000,29.575000,19.300000,0.000000,0.000000,14.075000,28.700000,0.000000
50%,33.218750,-6.687500,223.500000,33.350000,20.650000,0.000000,0.000000,17.300000,34.400000,1.000000
75%,34.187500,-5.546875,544.250000,36.525000,22.225000,0.000000,0.000000,20.900000,41.000000,2.000000
max,35.812500,-1.937500,1466.000000,42.300000,27.900000,4.600000,45.000000,33.800000,78.800000,80.000000


In [28]:
meteo_df["time"].unique()

<DatetimeArray>
['2026-09-20 00:00:00', '2026-09-21 00:00:00', '2026-09-22 00:00:00']
Length: 3, dtype: datetime64[us]

In [29]:
meteo_df[meteo_df["temperature_2m_max"]<meteo_df["temperature_2m_min"]]

,latitude,longitude,elevation,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code


In [30]:
meteo_df[meteo_df["wind_gusts_10m_max"]<meteo_df["wind_speed_10m_max"]]

,latitude,longitude,elevation,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code


3. effectuer la jointure entre les villes et les données météo.

In [51]:
meteo_par_villes_df = meteo_df.merge(ma_df, left_index=True, right_index=True)
meteo_par_villes_df.head()

,latitude,longitude,elevation,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code,city,lat,lng,country
0,33.5625,-7.6250,23.0,2026-09-20,30.3,20.5,0.0,0,10.1,29.5,0,Casablanca,33.5992,-7.6200,Morocco
0,33.5625,-7.6250,23.0,2026-09-21,31.3,21.6,0.0,0,11.5,34.2,0,Casablanca,33.5992,-7.6200,Morocco
0,33.5625,-7.6250,23.0,2026-09-22,29.8,19.8,0.0,0,9.6,28.8,0,Casablanca,33.5992,-7.6200,Morocco
1,35.7500,-5.8125,30.0,2026-09-20,27.7,23.7,0.0,0,33.8,78.8,0,Tangier,35.7767,-5.8039,Morocco
1,35.7500,-5.8125,30.0,2026-09-21,28.6,22.8,0.0,0,27.7,64.1,0,Tangier,35.7767,-5.8039,Morocco


In [52]:
meteo_par_villes_df.reset_index(inplace=True, drop=True)
meteo_par_villes_df.drop(columns=["lat", "lng"], inplace=True)
meteo_par_villes_df.head()

,latitude,longitude,elevation,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code,city,country
0,33.5625,-7.6250,23.0,2026-09-20,30.3,20.5,0.0,0,10.1,29.5,0,Casablanca,Morocco
1,33.5625,-7.6250,23.0,2026-09-21,31.3,21.6,0.0,0,11.5,34.2,0,Casablanca,Morocco
2,33.5625,-7.6250,23.0,2026-09-22,29.8,19.8,0.0,0,9.6,28.8,0,Casablanca,Morocco
3,35.7500,-5.8125,30.0,2026-09-20,27.7,23.7,0.0,0,33.8,78.8,0,Tangier,Morocco
4,35.7500,-5.8125,30.0,2026-09-21,28.6,22.8,0.0,0,27.7,64.1,0,Tangier,Morocco


4. stocker les données nettoyées dans silver/.

In [ ]:
try:
    os.mkdir(path="./silver")
    meteo_par_villes_df.to_csv(
        path_or_buf="./silver/cleaned_meteo_per_town.csv", 
        index=False
    )
except FileExistsError:
    meteo_par_villes_df.to_csv(
        path_or_buf="./silver/cleaned_meteo_per_town.csv", 
        index=False
    )
except Exception as e:
    print(e)

## Étape 3 — Feature Engineering / Gold

1. catégories de température.

In [53]:
meteo_par_villes_df[
    [column for column in meteo_par_villes_df.columns.to_list() if column.__contains__("temperature")]
]

,temperature_2m_max,temperature_2m_min
0,30.3,20.5
1,31.3,21.6
2,29.8,19.8
3,27.7,23.7
4,28.6,22.8
...,...,...
355,35.1,22.2
356,37.3,20.5
357,36.9,19.0
358,38.8,27.4


In [54]:
Zero_absolu = -273.15
Ebullition = +99.9839

temperature_bins = [Zero_absolu, 0, 35, 40, Ebullition]
temperature_labels = ["freezing", "normal", "heat", "extreme heat"]

meteo_par_villes_df["temperature_2m_max_risk"] = pd.cut(
    meteo_par_villes_df.temperature_2m_max,
    bins=temperature_bins,
    labels=temperature_labels,
    include_lowest=True,
    ordered=True
)

meteo_par_villes_df["temperature_2m_min_risk"] = pd.cut(
    meteo_par_villes_df.temperature_2m_min,
    bins=temperature_bins,
    labels=temperature_labels,
    include_lowest=True,
    ordered=True
)

meteo_par_villes_df.head()

,latitude,longitude,elevation,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code,city,country,temperature_2m_max_risk,temperature_2m_min_risk
0,33.5625,-7.6250,23.0,2026-09-20,30.3,20.5,0.0,0,10.1,29.5,0,Casablanca,Morocco,normal,normal
1,33.5625,-7.6250,23.0,2026-09-21,31.3,21.6,0.0,0,11.5,34.2,0,Casablanca,Morocco,normal,normal
2,33.5625,-7.6250,23.0,2026-09-22,29.8,19.8,0.0,0,9.6,28.8,0,Casablanca,Morocco,normal,normal
3,35.7500,-5.8125,30.0,2026-09-20,27.7,23.7,0.0,0,33.8,78.8,0,Tangier,Morocco,normal,normal
4,35.7500,-5.8125,30.0,2026-09-21,28.6,22.8,0.0,0,27.7,64.1,0,Tangier,Morocco,normal,normal


In [55]:
meteo_par_villes_df.columns

Index(['latitude', 'longitude', 'elevation', 'time', 'temperature_2m_max',
       'temperature_2m_min', 'precipitation_sum',
       'precipitation_probability_max', 'wind_speed_10m_max',
       'wind_gusts_10m_max', 'weather_code', 'city', 'country',
       'temperature_2m_max_risk', 'temperature_2m_min_risk'],
      dtype='str')

2. catégories de précipitations.

In [56]:
meteo_par_villes_df[
    [column for column in meteo_par_villes_df.columns.to_list() if column.__contains__("precipitation")]
]

,precipitation_sum,precipitation_probability_max
0,0.0,0
1,0.0,0
2,0.0,0
3,0.0,0
4,0.0,0
...,...,...
355,0.0,0
356,0.0,0
357,0.0,0
358,0.0,2


In [57]:
proba_bins = [0, 30, 60, 100]
proba_labels = ["low", "medium", "high"]

meteo_par_villes_df["precipitation_probability_risk"] = pd.cut(
    meteo_par_villes_df.precipitation_probability_max,
    bins=proba_bins,
    labels=proba_labels,
    include_lowest=True,
    ordered=True
)

sum_bins = [0, 2, 10, 25, np.inf]
sum_labels = ["low", "medium", "high", "critical"]

meteo_par_villes_df["precipitation_sum_risk"] = pd.cut(
    meteo_par_villes_df.precipitation_sum,
    bins=sum_bins,
    labels=sum_labels,
    include_lowest=True,
    ordered=True
)

meteo_par_villes_df

,latitude,longitude,elevation,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code,city,country,temperature_2m_max_risk,temperature_2m_min_risk,precipitation_probability_risk,precipitation_sum_risk
0,33.56250,-7.625000,23.0,2026-09-20,30.3,20.5,0.0,0,10.1,29.5,0,Casablanca,Morocco,normal,normal,low,low
1,33.56250,-7.625000,23.0,2026-09-21,31.3,21.6,0.0,0,11.5,34.2,0,Casablanca,Morocco,normal,normal,low,low
2,33.56250,-7.625000,23.0,2026-09-22,29.8,19.8,0.0,0,9.6,28.8,0,Casablanca,Morocco,normal,normal,low,low
3,35.75000,-5.812500,30.0,2026-09-20,27.7,23.7,0.0,0,33.8,78.8,0,Tangier,Morocco,normal,normal,low,low
4,35.75000,-5.812500,30.0,2026-09-21,28.6,22.8,0.0,0,27.7,64.1,0,Tangier,Morocco,normal,normal,low,low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,32.12500,-9.062500,173.0,2026-09-21,35.1,22.2,0.0,0,14.0,27.0,2,Sebt Gzoula,Morocco,heat,normal,low,low
356,32.12500,-9.062500,173.0,2026-09-22,37.3,20.5,0.0,0,15.0,28.8,2,Sebt Gzoula,Morocco,heat,normal,low,low
357,26.18629,-10.559204,486.0,2026-09-20,36.9,19.0,0.0,0,14.4,24.5,0,Tifariti,Morocco,heat,normal,low,low
358,26.18629,-10.559204,486.0,2026-09-21,38.8,27.4,0.0,2,26.5,42.8,0,Tifariti,Morocco,heat,normal,low,low


3. catégories de vent.

In [58]:
meteo_par_villes_df[
    [column for column in meteo_par_villes_df.columns.to_list() if column.__contains__("wind")]
]

,wind_speed_10m_max,wind_gusts_10m_max
0,10.1,29.5
1,11.5,34.2
2,9.6,28.8
3,33.8,78.8
4,27.7,64.1
...,...,...
355,14.0,27.0
356,15.0,28.8
357,14.4,24.5
358,26.5,42.8


In [59]:
wind_bins = [0,50,70, np.inf]
wind_risk_label = ["low", "medium", "high"]

meteo_par_villes_df["wind_speed_risk"] = pd.cut(
    meteo_par_villes_df.wind_speed_10m_max, 
    bins=wind_bins, 
    labels=wind_risk_label,
    ordered=True
)

meteo_par_villes_df["wind_gusts_risk"] = pd.cut(
    meteo_par_villes_df.wind_gusts_10m_max, 
    bins=wind_bins,
    labels=wind_risk_label,
    ordered=True
)
meteo_par_villes_df

,latitude,longitude,elevation,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code,city,country,temperature_2m_max_risk,temperature_2m_min_risk,precipitation_probability_risk,precipitation_sum_risk,wind_speed_risk,wind_gusts_risk
0,33.56250,-7.625000,23.0,2026-09-20,30.3,20.5,0.0,0,10.1,29.5,0,Casablanca,Morocco,normal,normal,low,low,low,low
1,33.56250,-7.625000,23.0,2026-09-21,31.3,21.6,0.0,0,11.5,34.2,0,Casablanca,Morocco,normal,normal,low,low,low,low
2,33.56250,-7.625000,23.0,2026-09-22,29.8,19.8,0.0,0,9.6,28.8,0,Casablanca,Morocco,normal,normal,low,low,low,low
3,35.75000,-5.812500,30.0,2026-09-20,27.7,23.7,0.0,0,33.8,78.8,0,Tangier,Morocco,normal,normal,low,low,low,high
4,35.75000,-5.812500,30.0,2026-09-21,28.6,22.8,0.0,0,27.7,64.1,0,Tangier,Morocco,normal,normal,low,low,low,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,32.12500,-9.062500,173.0,2026-09-21,35.1,22.2,0.0,0,14.0,27.0,2,Sebt Gzoula,Morocco,heat,normal,low,low,low,low
356,32.12500,-9.062500,173.0,2026-09-22,37.3,20.5,0.0,0,15.0,28.8,2,Sebt Gzoula,Morocco,heat,normal,low,low,low,low
357,26.18629,-10.559204,486.0,2026-09-20,36.9,19.0,0.0,0,14.4,24.5,0,Tifariti,Morocco,heat,normal,low,low,low,low
358,26.18629,-10.559204,486.0,2026-09-21,38.8,27.4,0.0,2,26.5,42.8,0,Tifariti,Morocco,heat,normal,low,low,low,low


4. date.

In [60]:
meteo_par_villes_df[
    [column for column in meteo_par_villes_df.columns.to_list() if column.__contains__("time")]
]

,time
0,2026-09-20
1,2026-09-21
2,2026-09-22
3,2026-09-20
4,2026-09-21
...,...
355,2026-09-21
356,2026-09-22
357,2026-09-20
358,2026-09-21


In [61]:
today = datetime.date.today()
conditions = [
    meteo_par_villes_df["time"] == today.strftime("%Y-%m-%d"),
    meteo_par_villes_df["time"] == today.replace(day=today.day+1).strftime("%Y-%m-%d"),
]
choises = ["today", "tomorrow"]
meteo_par_villes_df["when"] = np.select(conditions, choises, default="after tomorrow")
meteo_par_villes_df

,latitude,longitude,elevation,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,weather_code,city,country,temperature_2m_max_risk,temperature_2m_min_risk,precipitation_probability_risk,precipitation_sum_risk,wind_speed_risk,wind_gusts_risk,when
0,33.56250,-7.625000,23.0,2026-09-20,30.3,20.5,0.0,0,10.1,29.5,0,Casablanca,Morocco,normal,normal,low,low,low,low,today
1,33.56250,-7.625000,23.0,2026-09-21,31.3,21.6,0.0,0,11.5,34.2,0,Casablanca,Morocco,normal,normal,low,low,low,low,tomorrow
2,33.56250,-7.625000,23.0,2026-09-22,29.8,19.8,0.0,0,9.6,28.8,0,Casablanca,Morocco,normal,normal,low,low,low,low,after tomorrow
3,35.75000,-5.812500,30.0,2026-09-20,27.7,23.7,0.0,0,33.8,78.8,0,Tangier,Morocco,normal,normal,low,low,low,high,today
4,35.75000,-5.812500,30.0,2026-09-21,28.6,22.8,0.0,0,27.7,64.1,0,Tangier,Morocco,normal,normal,low,low,low,medium,tomorrow
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,32.12500,-9.062500,173.0,2026-09-21,35.1,22.2,0.0,0,14.0,27.0,2,Sebt Gzoula,Morocco,heat,normal,low,low,low,low,tomorrow
356,32.12500,-9.062500,173.0,2026-09-22,37.3,20.5,0.0,0,15.0,28.8,2,Sebt Gzoula,Morocco,heat,normal,low,low,low,low,after tomorrow
357,26.18629,-10.559204,486.0,2026-09-20,36.9,19.0,0.0,0,14.4,24.5,0,Tifariti,Morocco,heat,normal,low,low,low,low,today
358,26.18629,-10.559204,486.0,2026-09-21,38.8,27.4,0.0,2,26.5,42.8,0,Tifariti,Morocco,heat,normal,low,low,low,low,tomorrow


5. autres indicateurs pertinents.

6. Weather Risk Score
<br>
Créer un score de risque météorologique de 0 à 100 permettant d'identifier les conditions potentiellement défavorables.

In [62]:
high_risk = 100
risk_factors = [factor for factor in meteo_par_villes_df.columns.to_list() if factor.__contains__("risk")]
risk_factors

['temperature_2m_max_risk',
 'temperature_2m_min_risk',
 'precipitation_probability_risk',
 'precipitation_sum_risk',
 'wind_speed_risk',
 'wind_gusts_risk']

In [63]:
factor_score = 100/len(risk_factors)
factor_score

16.666666666666668

In [64]:
factor_categories = {
    'temperature_2m_max_risk': temperature_labels,
    'temperature_2m_min_risk': temperature_labels,
    'precipitation_probability_risk': proba_labels,
    'precipitation_sum_risk': sum_labels,
    'wind_speed_risk': wind_risk_label,
    'wind_gusts_risk': wind_risk_label
}

factor_distributed_score = {}

for factor in risk_factors:
    factor_distributed_score[factor] = np.linspace(
        0, 
        factor_score,
        len(factor_categories[factor]),
        endpoint=True,
    )

factor_distributed_score

{'temperature_2m_max_risk': array([ 0.        ,  5.55555556, 11.11111111, 16.66666667]),
 'temperature_2m_min_risk': array([ 0.        ,  5.55555556, 11.11111111, 16.66666667]),
 'precipitation_probability_risk': array([ 0.        ,  8.33333333, 16.66666667]),
 'precipitation_sum_risk': array([ 0.        ,  5.55555556, 11.11111111, 16.66666667]),
 'wind_speed_risk': array([ 0.        ,  8.33333333, 16.66666667]),
 'wind_gusts_risk': array([ 0.        ,  8.33333333, 16.66666667])}

In [65]:
temperature_risk_scores = {
    "freezing": factor_distributed_score["temperature_2m_max_risk"][2], 
    "normal": factor_distributed_score["temperature_2m_max_risk"][0], 
    "heat": factor_distributed_score["temperature_2m_max_risk"][1], 
    "extreme heat": factor_distributed_score["temperature_2m_max_risk"][3]
}

temperature_risk_scores

{'freezing': np.float64(11.111111111111112),
 'normal': np.float64(0.0),
 'heat': np.float64(5.555555555555556),
 'extreme heat': np.float64(16.666666666666668)}

In [66]:
precipitation_proba_risk_scores = {
    x:factor_distributed_score["precipitation_probability_risk"][i]
    for i,x in enumerate(proba_labels)
}

precipitation_proba_risk_scores

{'low': np.float64(0.0),
 'medium': np.float64(8.333333333333334),
 'high': np.float64(16.666666666666668)}

In [67]:
precipitation_sum_risk_scores = {
    x:factor_distributed_score["precipitation_sum_risk"][i]
    for i,x in enumerate(sum_labels)
}

precipitation_sum_risk_scores

{'low': np.float64(0.0),
 'medium': np.float64(5.555555555555556),
 'high': np.float64(11.111111111111112),
 'critical': np.float64(16.666666666666668)}

In [68]:
wind_risk_scores = {
    x:factor_distributed_score["wind_speed_risk"][i]
    for i,x in enumerate(wind_risk_label)
}

wind_risk_scores

{'low': np.float64(0.0),
 'medium': np.float64(8.333333333333334),
 'high': np.float64(16.666666666666668)}

In [69]:
meteo_par_villes_df["risk_score"] = (
    meteo_par_villes_df.temperature_2m_max_risk.map(temperature_risk_scores).astype(float)
    + meteo_par_villes_df.temperature_2m_min_risk.map(temperature_risk_scores).astype(float)
    + meteo_par_villes_df.precipitation_probability_risk.map(precipitation_proba_risk_scores).astype(float)
    + meteo_par_villes_df.precipitation_sum_risk.map(precipitation_sum_risk_scores).astype(float)
    + meteo_par_villes_df.wind_speed_risk.map(wind_risk_scores).astype(float)
    + meteo_par_villes_df.wind_gusts_risk.map(wind_risk_scores).astype(float)
)

meteo_par_villes_df.head()

,latitude,longitude,elevation,time,temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,wind_speed_10m_max,wind_gusts_10m_max,...,city,country,temperature_2m_max_risk,temperature_2m_min_risk,precipitation_probability_risk,precipitation_sum_risk,wind_speed_risk,wind_gusts_risk,when,risk_score
0,33.5625,-7.6250,23.0,2026-09-20,30.3,20.5,0.0,0,10.1,29.5,...,Casablanca,Morocco,normal,normal,low,low,low,low,today,0.000000
1,33.5625,-7.6250,23.0,2026-09-21,31.3,21.6,0.0,0,11.5,34.2,...,Casablanca,Morocco,normal,normal,low,low,low,low,tomorrow,0.000000
2,33.5625,-7.6250,23.0,2026-09-22,29.8,19.8,0.0,0,9.6,28.8,...,Casablanca,Morocco,normal,normal,low,low,low,low,after tomorrow,0.000000
3,35.7500,-5.8125,30.0,2026-09-20,27.7,23.7,0.0,0,33.8,78.8,...,Tangier,Morocco,normal,normal,low,low,low,high,today,16.666667
4,35.7500,-5.8125,30.0,2026-09-21,28.6,22.8,0.0,0,27.7,64.1,...,Tangier,Morocco,normal,normal,low,low,low,medium,tomorrow,8.333333


Vous devrez justifier :
* les variables utilisées.
* les seuils.
* la méthode de calcul.

9. Charger les données finales dans PostgreSQL.<br>
Le modèle devra permettre de gérer au minimum :
* les villes et leurs coordonnées.
* les prévisions météorologiques.
* le risk_score.<br>
Vous devrez également prévoir une stratégie pour éviter les doublons lors des nouvelles exécutions du pipeline, car les prévisions peuvent être mises à jour.

In [79]:
from sqlalchemy.engine import URL
from dotenv import load_dotenv
load_dotenv()

url = URL.create(
    "postgresql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(url)

In [81]:
with engine.connect() as connexion:
    connexion.execute(text("""
CREATE TABLE IF NOT EXISTS ville (
    id SERIAL PRIMARY KEY,
    city VARCHAR(100) NOT NULL,
    country VARCHAR(100) NOT NULL,

    UNIQUE (city, country)
);

CREATE TABLE IF NOT EXISTS datapoint (
    id SERIAL PRIMARY KEY,
    ville_id INTEGER NOT NULL REFERENCES ville(id),

    longitude DECIMAL(9,6) NOT NULL,
    latitude DECIMAL(9,6) NOT NULL,
    elevation DECIMAL(8,2),

    UNIQUE (longitude, latitude)
);

CREATE TABLE IF NOT EXISTS prevision_meteo (
    id SERIAL PRIMARY KEY,
    datapoint_id INTEGER NOT NULL REFERENCES datapoint(id),
    time DATE NOT NULL,
    temperature_2m_max DECIMAL(5,2),
    temperature_2m_min DECIMAL(5,2),
    precipitation_sum DECIMAL(6,2),
    precipitation_probability_max DECIMAL(5,2),
    wind_speed_10m_max DECIMAL(6,2),
    wind_gusts_10m_max DECIMAL(6,2),
    weather_code SMALLINT,
    UNIQUE (datapoint_id, time)
);

CREATE TABLE IF NOT EXISTS weather_risk (
    id SERIAL PRIMARY KEY,
    prevision_id INTEGER NOT NULL REFERENCES prevision_meteo(id),
    temperature_2m_max_risk DECIMAL(5,2),
    temperature_2m_min_risk DECIMAL(5,2),
    precipitation_probability_risk DECIMAL(5,2),
    precipitation_sum_risk DECIMAL(5,2),
    wind_speed_risk DECIMAL(5,2),
    wind_gusts_risk DECIMAL(5,2),
    risk_score DECIMAL(10,2),
    UNIQUE (prevision_id)
);
"""))
    
    connexion.commit()

In [82]:
with engine.connect() as connexion:
    meteo_par_villes_df[["city", "country"]].to_sql(
        "ville", 
        connexion,
        if_exists='append',
        index=False,
        chunksize=100,
    )

DatabaseError: Execution failed on sql 'INSERT INTO ville (city, country) VALUES (:city, :country)': (psycopg2.errors.UniqueViolation) ERREUR:  la valeur d'une clé dupliquée rompt la contrainte unique « ville_city_country_key »
DETAIL:  La clé « (city, country)=(Casablanca, Morocco) » existe déjà.

[SQL: INSERT INTO ville (city, country) VALUES (%(city__0)s, %(country__0)s), (%(city__1)s, %(country__1)s), (%(city__2)s, %(country__2)s), (%(city__3)s, %(country__3)s), (%(city__4)s, %(country__4)s), (%(city__5)s, %(country__5)s), (%(city__6)s, %(country ... 2969 characters truncated ... ), (%(city__97)s, %(country__97)s), (%(city__98)s, %(country__98)s), (%(city__99)s, %(country__99)s)]
[parameters: {'city__0': 'Casablanca', 'country__0': 'Morocco', 'city__1': 'Casablanca', 'country__1': 'Morocco', 'city__2': 'Casablanca', 'country__2': 'Morocco', 'city__3': 'Tangier', 'country__3': 'Morocco', 'city__4': 'Tangier', 'country__4': 'Morocco', 'city__5': 'Tangier', 'country__5': 'Morocco', 'c

9. Bonus:
<br>
conserver l'historique des différentes prévisions.

## Étape 4 — Analyse SQL
Réaliser au minimum 5 requêtes SQL répondant à des questions métier.

1. Quelles villes auront les températures les plus élevées ?

2. Quelles villes auront les plus fortes précipitations ?

3. Quelles villes présentent le risque moyen le plus élevé ?

4. Quelles périodes présentent le risque maximal ?

5. Pour chaque ville, quelle période présente le plus grand risque ?

6. Bonus:
sous-requêtes, fonctions de fenêtrage.

## Étape 5 — Dashboard Streamlit
Créer un dashboard connecté à PostgreSQL permettant de visualiser les prévisions et les risques.
<br>
Le dashboard doit permettre de répondre rapidement à la question :
Où et quand faut-il être particulièrement vigilant dans les prochains jours ?

1. KPI (Key Performance Indicator):
* nombre de villes.
* température maximale.
* précipitations maximales.
* nombre de périodes à risque.
* ville présentant le risque le plus élevé.

2. Filtres:
<br>
Permettre de filtrer notamment par : 
* ville. 
* date. 
* période. 
* niveau de risque.

## Étape 6 — Orchestration & automatisation (Airflow)

1. Définir un
DAG Airflow
qui automatise l'ensemble du pipeline :
* Extraction depuis l'API.
* Nettoyage / transformation.
* Feature engineering & chargement dans PostgreSQL.
* (Rafraîchissement des données pour le dashboard).

2. Planifier une exécution automatique (ex. quotidienne) et gérer les échecs (retries).

3. Conteneuriser le projet avec Docker Compose (Postgres + Airflow + Streamlit).

4. Bonus:
* Ajouter du logging structuré et des alertes en cas d'échec du DAG.
* Historique des prévisions.
* Pipeline incrémental.
* Contrôles de qualité.